<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 10B: *Fire Damage Feature Ablation*
##### Version Number: 4.0
---
### Contents  
> *Build Models*\
> *SHAP*\
> *Set Ablation*
---
### Notes
---
### Inputs
---
### Outputs 
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import json

# Modeling libraries
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import train_test_split

---

###  Load Data

In [3]:
X_damage = pd.read_csv('../data/processed/X_damage.csv')
y_damage = pd.read_csv('../data/processed/y_damage.csv').squeeze()  # Load as Series
details_damage = pd.read_csv('../data/processed/details_damage.csv')

pal_details = pd.read_csv('../data/processed/pal_details.csv')
pal_X = pd.read_csv('../data/processed/pal_X.csv')
pal_y = pd.read_csv('../data/processed/pal_y.csv')

best_strategy = pd.read_csv('../data/processed/damage_best_strategy.csv')

with open('model_parameters_ignition.json', 'r') as f:
    model_parameters = json.load(f)

with open('feature_sets.json', 'r') as f:
    feature_sets = json.load(f)

## Subset

In [4]:
reform = pd.concat([X_damage,y_damage], axis=1)
subset = subset_df(reform, 'Target_Damage', 500)

y = subset['Target_Damage']
X = subset.drop(columns='Target_Damage')

In [5]:
X_rf, y_rf = apply_balancing('RF', best_strategy, X, y)
X_xgb, y_xgb = apply_balancing('XGB', best_strategy, X, y)

## Build Models

In [6]:
RF_parameters = model_parameters['Random Forest']
XGB_parameters = model_parameters['XGBoost']

# Build Final tuned models
damage_xgb = xgb.XGBClassifier(**XGB_parameters)
damage_rf = RandomForestClassifier(**RF_parameters)

display(RF_parameters)
display(XGB_parameters)

{'n_estimators': 150,
 'max_depth': 15,
 'min_samples_split': 10,
 'max_features': 'log2',
 'class_weight': 'balanced'}

{'objective': 'multi:softmax',
 'num_class': 3,
 'n_estimators': 150,
 'max_depth': 3,
 'learning_rate': 0.4,
 'verbosity': 0}

In [7]:
models = {
    "XGB": damage_xgb, 
    "RF": damage_rf
}

## SHAP

### XGB Damage SHAP

In [8]:
xgb_top_10 = get_shap(damage_xgb, X_xgb, y_xgb)
xgb_top_10


,0,1
0,intermix_zone,0.457668
1,influence_zone,0.421615
2,Daily Minimum Air Temperature,0.420325
3,1000-hour Dead Fuel Moisture,0.387855
4,dominant_province_description_American Semi-De...,0.296135
5,Vapor Pressure Deficit 7 Day Avg,0.264723
6,Palmer Drought Severity Index,0.247867
7,road_length_meters,0.245365
8,SPEI 30-Day,0.226122
9,Season_Spring,0.223408


### RF Damage SHAP

In [9]:
rf_top_10 = get_shap(damage_rf, X_rf, y_rf)
rf_top_10 


,0,1
0,intermix_zone,0.027066
1,influence_zone,0.023255
2,Vapor Pressure Deficit 7 Day Avg,0.021840
3,road_length_meters,0.020391
4,1000-hour Dead Fuel Moisture,0.019514
5,interface_zone,0.018503
6,road_density,0.016221
7,Daily Minimum Air Temperature,0.015934
8,Daily Maximum Air Temperature 7 Day Avg,0.014087
9,Daily Maximum Air Temperature,0.013103


## Set Ablation

In [10]:
for set_name, columns in feature_sets.items():
    print(f"{set_name}: {columns}")
    print("\n")

Water Demand: ['Actual Evapotranspiration', 'Solar Radiation', 'Solar Radiation 7 Day Avg', 'Daily Minimum Air Temperature', 'Daily Maximum Air Temperature', 'Daily Maximum Air Temperature 7 Day Avg', 'Daily Minimum Air Temperature 7 Day Avg', 'Vapor Pressure Deficit', 'Vapor Pressure Deficit 7 Day Avg', 'Wind Speed', 'Wind Speed 7 Day Avg']


Water Supply: ['Precipitation', 'Precipitation 7 Day Avg', 'Maximum Relative Humidity', 'Minimum Relative Humidity', 'Maximum Relative Humidity 7 Day Avg', 'Minimum Relative Humidity 7 Day Avg', 'Specific Humidity', '100-hour Dead Fuel Moisture', '1000-hour Dead Fuel Moisture']


Water Supply Indexes: ['SPI 30-Day', 'SPI 180-Day', 'SPEI 30-Day', 'SPEI 90-Day', 'SPEI 180-Day', 'Palmer Drought Severity Index']


Fire Danger: ['Burning Index', 'Energy Release Component', 'Santa_Ana_Score']


Social: ['total_housing', 'total_population', 'housing_density', 'population_density', 'median_income']


Elevation: ['elevation_range', 'elevation_mean', 'elev

In [11]:
compare = []

compare.append(compare_model(damage_xgb, X, y, best_strategy,
                                     name = 'XGB', test_set = 'Full'))

compare.append(compare_model(damage_rf, X, y, best_strategy,
                                     name = 'RF', test_set = 'Full'))

for set_name, set_list in feature_sets.items():
    for model_name, model in models.items():
        print("Testing " + f"{model_name}: {set_name}")
        X_section = X.drop(columns = set_list)
        compare.append(compare_model(model, X_section, y, best_strategy,
                                     name = model_name, test_set = set_name))

Testing XGB: Water Demand
Testing RF: Water Demand
Testing XGB: Water Supply
Testing RF: Water Supply
Testing XGB: Water Supply Indexes
Testing RF: Water Supply Indexes
Testing XGB: Fire Danger
Testing RF: Fire Danger
Testing XGB: Social
Testing RF: Social
Testing XGB: Elevation
Testing RF: Elevation
Testing XGB: WUI
Testing RF: WUI
Testing XGB: Ecoregion
Testing RF: Ecoregion
Testing XGB: Land Cover
Testing RF: Land Cover
Testing XGB: Interactions
Testing RF: Interactions
Testing XGB: Wind Slope
Testing RF: Wind Slope
Testing XGB: Others
Testing RF: Others
Testing XGB: Coded Ecoregions
Testing RF: Coded Ecoregions
Testing XGB: Coded Seasons
Testing RF: Coded Seasons


In [12]:
comparisons = pd.DataFrame(compare)

In [13]:
XGB = comparisons[comparisons['Model'] == 'XGB'] 
RF = comparisons[comparisons['Model'] == 'RF'] 

In [14]:
full_XGB = XGB.loc[XGB['Test Set']=='Full','Macro F1'].values[0]
full_RF = RF.loc[RF['Test Set']=='Full','Macro F1'].values[0]

In [15]:
XGB['Macro F1 Percent Difference'] = ((full_XGB - XGB['Macro F1']) / full_XGB) * 100
RF['Macro F1 Percent Difference'] = ((full_RF - RF['Macro F1']) / full_RF) * 100

C:\Users\dusti\AppData\Local\Temp\ipykernel_3448\2524239291.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  XGB['Macro F1 Percent Difference'] = ((full_XGB - XGB['Macro F1']) / full_XGB) * 100
C:\Users\dusti\AppData\Local\Temp\ipykernel_3448\2524239291.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  RF['Macro F1 Percent Difference'] = ((full_RF - RF['Macro F1']) / full_RF) * 100


In [16]:
RF

,Test Set,Model,Weighted F1,Macro F1,High Risk Recall,Macro F1 Percent Difference
1,Full,RF,0.890089,0.889291,0.889908,0.000000
3,Water Demand,RF,0.884817,0.883718,0.908257,0.626732
5,Water Supply,RF,0.890089,0.889291,0.889908,0.000000
7,Water Supply Indexes,RF,0.895119,0.894406,0.889908,-0.575128
9,Fire Danger,RF,0.895119,0.894406,0.889908,-0.575128
11,Social,RF,0.890155,0.889458,0.880734,-0.018766
13,Elevation,RF,0.890155,0.889458,0.880734,-0.018766
15,WUI,RF,0.890155,0.889458,0.880734,-0.018766
17,Ecoregion,RF,0.885188,0.884512,0.871560,0.537439
19,Land Cover,RF,0.895119,0.894406,0.889908,-0.575128


In [17]:
XGB

,Test Set,Model,Weighted F1,Macro F1,High Risk Recall,Macro F1 Percent Difference
0,Full,XGB,0.894950,0.894044,0.908257,0.000000
2,Water Demand,XGB,0.879733,0.878530,0.908257,1.735209
4,Water Supply,XGB,0.889755,0.888653,0.917431,0.602996
6,Water Supply Indexes,XGB,0.879879,0.878788,0.899083,1.706389
8,Fire Danger,XGB,0.889889,0.888889,0.908257,0.576578
10,Social,XGB,0.905041,0.904309,0.908257,-1.148141
12,Elevation,XGB,0.904849,0.903941,0.926606,-1.107016
14,WUI,XGB,0.894833,0.893829,0.917431,0.023967
16,Ecoregion,XGB,0.904849,0.903941,0.926606,-1.107016
18,Land Cover,XGB,0.904849,0.903941,0.926606,-1.107016
